# NGLab Tutorial #8: Multi-Agent Simulation

Explore competitive multi-agent environments where multiple trading agents interact.

## Learning Objectives

1. Create a multi-agent arena
2. Implement different agent strategies
3. Analyze strategic emergence
4. Visualize Nash equilibria

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

np.random.seed(42)
print("Libraries loaded!")

## 1. Agent Types

We'll implement 4 agent archetypes:
1. **Market Maker**: Provides liquidity on both sides
2. **Trend Follower**: Buys uptrends, sells downtrends
3. **Mean Reverter**: Fades extremes
4. **Random Trader**: Noise trader baseline

In [ ]:
class BaseAgent:
    def __init__(self, name, initial_cash=10000, initial_position=0):
        self.name = name
        self.cash = initial_cash
        self.position = initial_position
        self.pnl_history = []
    
    def act(self, observation):
        """Returns action: 0=hold, 1=buy, 2=sell"""
        raise NotImplementedError
    
    def get_portfolio_value(self, price):
        return self.cash + self.position * price

class MarketMaker(BaseAgent):
    def __init__(self, *args, **kwargs):
        super().__init__("MarketMaker", *args, **kwargs)
    
    def act(self, observation):
        # Alternate buy/sell to provide liquidity
        return np.random.choice([1, 2])

class TrendFollower(BaseAgent):
    def __init__(self, *args, **kwargs):
        super().__init__("TrendFollower", *args, **kwargs)
    
    def act(self, observation):
        # Buy if recent trend is up, sell if down
        if len(observation) < 2:
            return 0
        
        trend = observation[-1] - observation[-10] if len(observation) >= 10 else 0
        if trend > 0:
            return 1  # Buy
        elif trend < 0:
            return 2  # Sell
        return 0

class MeanReverter(BaseAgent):
    def __init__(self, *args, **kwargs):
        super().__init__("MeanReverter", *args, **kwargs)
    
    def act(self, observation):
        # Fade deviations from mean
        if len(observation) < 20:
            return 0
        
        mean = np.mean(observation[-20:])
        current = observation[-1]
        
        if current > mean * 1.01:  # Above mean → sell
            return 2
        elif current < mean * 0.99:  # Below mean → buy
            return 1
        return 0

class RandomAgent(BaseAgent):
    def __init__(self, *args, **kwargs):
        super().__init__("RandomAgent", *args, **kwargs)
    
    def act(self, observation):
        return np.random.choice([0, 1, 2])

print("Agent classes defined!")

## 2. Multi-Agent Arena

In [ ]:
class MultiAgentArena:
    def __init__(self, agents, initial_price=50000, volatility=100):
        self.agents = agents
        self.price = initial_price
        self.volatility = volatility
        self.price_history = [initial_price]
        self.step_count = 0
    
    def step(self):
        """Execute one timestep of multi-agent trading."""
        # Get actions from all agents
        actions = {}
        for agent in self.agents:
            obs = np.array(self.price_history)
            actions[agent.name] = agent.act(obs)
        
        # Aggregate order flow
        buy_pressure = sum(1 for a in actions.values() if a == 1)
        sell_pressure = sum(1 for a in actions.values() if a == 2)
        
        # Update price based on imbalance
        imbalance = buy_pressure - sell_pressure
        price_change = imbalance * (self.volatility / len(self.agents))
        
        # Add random walk component
        price_change += np.random.randn() * self.volatility * 0.5
        
        self.price += price_change
        self.price = max(self.price, 1000)  # Floor
        self.price_history.append(self.price)
        
        # Execute trades for agents
        trade_size = 0.01  # 0.01 BTC per trade
        for agent in self.agents:
            action = actions[agent.name]
            
            if action == 1 and agent.cash >= self.price * trade_size:
                # Buy
                agent.position += trade_size
                agent.cash -= self.price * trade_size
            elif action == 2 and agent.position >= trade_size:
                # Sell
                agent.position -= trade_size
                agent.cash += self.price * trade_size
            
            # Record PnL
            pnl = agent.get_portfolio_value(self.price) - 10000
            agent.pnl_history.append(pnl)
        
        self.step_count += 1

print("MultiAgentArena defined!")

## 3. Run Tournament

In [ ]:
# Create agents
agents = [
    MarketMaker("MM_1"),
    MarketMaker("MM_2"),
    TrendFollower("TF_1"),
    MeanReverter("MR_1"),
    RandomAgent("Noise_1"),
]

# Create arena
arena = MultiAgentArena(agents)

# Run simulation
n_steps = 1000
for _ in range(n_steps):
    arena.step()

print(f"\n=== Tournament Results ({n_steps} steps) ===")
for agent in agents:
    final_value = agent.get_portfolio_value(arena.price)
    pnl = final_value - 10000
    returns = (final_value / 10000 - 1) * 100
    print(f"{agent.name:15s}: ${final_value:,.2f} (PnL: ${pnl:+,.2f}, Return: {returns:+.2f}%)")

## 4. Visualize Competition

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Price evolution
ax1.plot(arena.price_history, linewidth=2, color='black', alpha=0.7)
ax1.set_title('Market Price Over Time', fontsize=14, fontweight='bold')
ax1.set_ylabel('Price (USD)')
ax1.grid(True, alpha=0.3)

# Agent PnL
colors = plt.cm.tab10(np.linspace(0, 1, len(agents)))
for agent, color in zip(agents, colors):
    ax2.plot(agent.pnl_history, label=agent.name, linewidth=2, color=color)

ax2.axhline(y=0, color='red', linestyle='--', alpha=0.5)
ax2.set_title('Agent PnL Evolution', fontsize=14, fontweight='bold')
ax2.set_xlabel('Step')
ax2.set_ylabel('PnL (USD)')
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Strategic Analysis

Analyze which strategies emerged as dominant:

In [ ]:
# Group by strategy type
strategy_performance = defaultdict(list)
for agent in agents:
    strategy = agent.__class__.__name__
    final_value = agent.get_portfolio_value(arena.price)
    returns = (final_value / 10000 - 1) * 100
    strategy_performance[strategy].append(returns)

# Compute statistics
print("\n=== Strategy Performance ===")
for strategy, returns_list in strategy_performance.items():
    mean_return = np.mean(returns_list)
    std_return = np.std(returns_list) if len(returns_list) > 1 else 0
    print(f"{strategy:20s}: {mean_return:+.2f}% ± {std_return:.2f}%")

## Summary

In this notebook, you learned:

✅ Multi-agent environment design  
✅ Different trading strategies (MM, TF, MR)  
✅ Strategic emergence and competition  
✅ Performance analysis across strategies  

## Next Steps

Continue to **Notebook #9**: Backtesting Framework!

---